# Week10 Capstone project - Transparent logged ucb

## Strategy
For this round, the policy emphasises **transparency**. The notebook keeps the search local and human-clean, but logs every important decision.

The method is:
1. load the initial `.npy` data and the historical submission record from `Week10.csv`
2. use the **best recent weekly point**, the **second-best recent weekly point**, and the **latest point** as anchors
3. generate a small, deterministic neighbourhood of candidate micro-edits around those anchors
4. fit a GP surrogate and score candidates with a transparent UCB-like rule
5. save a per-function decision log to `logs/week10_decisions.json`

Scoring rule:

$$\text{score}(x)=\mu(x)+\beta(d)\sigma(x)-0.15\,\text{dist}(x, \text{best})-0.05\,\text{dist}(x, \text{latest})+0.02\,\text{novelty}(x)$$

with

$$\beta(d)=0.10+0.05d$$

This keeps the process reproducible and makes it easy for another researcher to inspect which anchors, candidate pool, and score decomposition led to the final choice.


In [2]:
import json
import numpy as np
import pandas as pd
import warnings
from pathlib import Path
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from sklearn.exceptions import ConvergenceWarning


/Users/sundarid/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
data_dir = Path('data')
logs_dir = data_dir / 'logs'
logs_dir.mkdir(exist_ok=True)
history = pd.read_csv(data_dir / 'weekly_results/Week10.csv')
history.tail(8)


,week,function,d,x1,x2,x3,x4,x5,x6,x7,x8,y
64,9,1,2,0.62800,0.63200,NaN,NaN,NaN,NaN,NaN,NaN,1.941930
65,9,2,2,0.49000,0.71000,NaN,NaN,NaN,NaN,NaN,NaN,0.542411
66,9,3,3,0.37200,0.56800,0.47200,NaN,NaN,NaN,NaN,NaN,-0.004842
67,9,4,4,0.39650,0.39350,0.35900,0.44050,NaN,NaN,NaN,NaN,0.217208
68,9,5,4,0.99999,0.00001,0.99999,0.99999,NaN,NaN,NaN,NaN,4440.106253
69,9,6,5,0.52500,0.17500,0.62500,0.87500,0.0250,NaN,NaN,NaN,-0.495826
70,9,7,6,0.01450,0.18150,0.28950,0.17250,0.3980,0.6660,NaN,NaN,2.099082
71,9,8,8,0.03480,0.43820,0.01180,0.32380,0.9905,0.0462,0.0978,0.7032,9.590284


In [4]:
def load_initial(function_id):
    X0 = np.load(data_dir / f'initial_data/F{function_id}_initial_inputs.npy')
    y0 = np.load(data_dir / f'initial_data/F{function_id}_initial_outputs.npy')
    return X0, y0

def load_combined(function_id, history_df):
    X0, y0 = load_initial(function_id)
    rows = history_df[history_df['function'] == function_id].sort_values('week')
    d = int(rows['d'].iloc[0])
    Xw = rows[[f'x{i}' for i in range(1, d + 1)]].to_numpy(dtype=float)
    yw = rows['y'].to_numpy(dtype=float)
    X = np.vstack([X0, Xw])
    y = np.concatenate([y0, yw])
    return X0, y0, Xw, yw, X, y, d, rows

def round_grid(x, grid):
    return np.clip(np.round(x / grid) * grid, 0.0, 1.0)

def format_point(x):
    return '-'.join(f'{float(v):.6f}' for v in x)


In [5]:
cfg = {
    1: {'grid': 0.001,   'step': 0.001},
    2: {'grid': 0.005,   'step': 0.005},
    3: {'grid': 0.001,   'step': 0.002},
    4: {'grid': 0.0005,  'step': 0.0005},
    5: {'grid': 0.00001, 'step': 0.00001},
    6: {'grid': 0.001,   'step': 0.001},
    7: {'grid': 0.0005,  'step': 0.0005},
    8: {'grid': 0.0001,  'step': 0.0001},
}

def beta_by_dimension(d):
    return 0.10 + 0.05 * d

def build_gp(seed, d):
    kernel = (
        ConstantKernel(1.0, (0.1, 10.0))
        * RBF(length_scale=np.ones(d), length_scale_bounds=(0.01, 1.5))
        + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
    )
    return GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        random_state=seed,
        n_restarts_optimizer=1,
    )


In [6]:
def build_candidates(best_x, latest_x, second_x, d, step, grid):
    candidates = []
    anchors = [
        best_x,
        latest_x,
        second_x,
        (best_x + latest_x) / 2.0,
        (2 * best_x + latest_x) / 3.0,
        (best_x + second_x) / 2.0,
    ]
    for a in anchors:
        candidates.append(round_grid(a, grid))

    # Single-axis micro-edits around best and latest.
    for anchor, multipliers in [(best_x, [1, -1, 2, -2]), (latest_x, [1, -1])]:
        for i in range(d):
            for m in multipliers:
                x = anchor.copy()
                x[i] += m * step
                candidates.append(round_grid(x, grid))

    return np.unique(np.array(candidates), axis=0)


In [7]:
def suggest_point(function_id, history_df, seed=1100):
    X0, y0, Xw, yw, X, y, d, rows = load_combined(function_id, history_df)

    recent_best_idx = int(np.argmax(yw))
    recent_second_idx = int(np.argsort(yw)[-2])
    best_x = Xw[recent_best_idx].copy()
    second_x = Xw[recent_second_idx].copy()
    latest_x = Xw[-1].copy()

    grid = cfg[function_id]['grid']
    step = cfg[function_id]['step']
    beta = beta_by_dimension(d)
    candidates = build_candidates(best_x, latest_x, second_x, d, step, grid)

    dist = np.sqrt(((candidates[:, None, :] - X[None, :, :]) ** 2).sum(axis=2)) / np.sqrt(d)
    min_dist = dist.min(axis=1)
    novelty_floor = grid / 2 if grid < 0.001 else 0.0005
    keep = min_dist >= novelty_floor
    candidates = candidates[keep]
    min_dist = min_dist[keep]

    gp = build_gp(seed + function_id, d)
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=ConvergenceWarning)
        gp.fit(X, y)

    mean, std = gp.predict(candidates, return_std=True)
    best_dist = np.sqrt(((candidates - best_x) ** 2).sum(axis=1)) / np.sqrt(d)
    latest_dist = np.sqrt(((candidates - latest_x) ** 2).sum(axis=1)) / np.sqrt(d)
    score = mean + beta * std - 0.15 * best_dist - 0.05 * latest_dist + 0.02 * min_dist

    top_idx = np.argsort(score)[-5:][::-1]
    chosen_idx = int(top_idx[0])
    x_best = candidates[chosen_idx]

    log = {
        'function': function_id,
        'dimension': d,
        'beta': float(beta),
        'grid': float(grid),
        'step': float(step),
        'recent_best_anchor': best_x.tolist(),
        'recent_second_anchor': second_x.tolist(),
        'latest_anchor': latest_x.tolist(),
        'chosen_point': x_best.tolist(),
        'chosen_point_formatted': format_point(x_best),
        'top_candidates': [
            {
                'point': candidates[i].tolist(),
                'formatted': format_point(candidates[i]),
                'score': float(score[i]),
                'mean': float(mean[i]),
                'std': float(std[i]),
                'dist_best': float(best_dist[i]),
                'dist_latest': float(latest_dist[i]),
                'novelty': float(min_dist[i]),
            }
            for i in top_idx
        ],
    }

    return x_best, log


In [ ]:
results = []
decision_logs = []
for function_id in range(1, 9):
    x, log = suggest_point(function_id, history)
    decision_logs.append(log)
    row = {'function': function_id, 'd': len(x)}
    for i, v in enumerate(x, start=1):
        row[f'x{i}'] = round(float(v), 6)
    results.append(row)
    print(f"Function {function_id}: {format_point(x)}")

week10_df = pd.DataFrame(results)
week10_df.to_csv('Week10_suggestions.csv', index=False)

with open(logs_dir / 'week10_decisions.json', 'w', encoding='utf-8') as f:
    json.dump(decision_logs, f, indent=2)

week10_df
